# Section 3 — KNN Imputation and Distribution Comparison (25 pts)

## Learning Objectives
- Apply `KNNImputer` with at least two different k values
- Compare KNN vs mean/median imputation
- Analyze distributional changes with plots and summary statistics

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
|      |          |                                |

## Tasks
- a) Select numerical columns for KNNImputer
- b) Apply KNN imputation with at least two k values
- c) Compare KNN with mean or median imputation
- d) Plot distributions before/after for ≥ 3 variables
- e) Compare mean, median, std, min, max, and IQR

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.config import RAW_DATA_PATH, MISSING_SENTINEL, KNN_K_VALUES, FIGURES_DIR
from src.load_data import load_raw_air_quality, build_datetime
from src.missing_values import replace_sentinel_with_nan, impute_basic
from src.knn_imputation import (
    select_numerical_columns,
    knn_impute,
    compare_distributions,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild the post-missing-value dataset so the notebook works on its own.
df = load_raw_air_quality(RAW_DATA_PATH)
df = build_datetime(df)
df = replace_sentinel_with_nan(df, sentinel=MISSING_SENTINEL)
display(df.head())

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Datetime
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-03-10 18:00:00
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-03-10 19:00:00
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-03-10 20:00:00
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-03-10 21:00:00
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-03-10 22:00:00


In [2]:
# Keep only the numerical columns that can be imputed with a distance-based method.
num_cols = select_numerical_columns(df)
display(pd.Series(num_cols, name='numerical_columns'))

0            CO(GT)
1       PT08.S1(CO)
2          NMHC(GT)
3          C6H6(GT)
4     PT08.S2(NMHC)
5           NOx(GT)
6      PT08.S3(NOx)
7           NO2(GT)
8      PT08.S4(NO2)
9       PT08.S5(O3)
10                T
11               RH
12               AH
Name: numerical_columns, dtype: str

In [3]:
# Compare a couple of K values so the choice is driven by the data, not by guesswork.
imputed = {}
for k in KNN_K_VALUES:
    imputed[f'knn_k{k}'] = knn_impute(df, num_cols, n_neighbors=k)

baseline_mean = impute_basic(df, columns=num_cols, strategy='mean')
baseline_median = impute_basic(df, columns=num_cols, strategy='median')

In [4]:
# Build a compact comparison table before plotting the distributions.
comparison_rows = []
for label, frame in {'mean': baseline_mean, 'median': baseline_median, **imputed}.items():
    comparison_rows.append({
        'method': label,
        'remaining_missing': int(frame[num_cols].isna().sum().sum()),
        'avg_mean': frame[num_cols].mean().mean(),
        'avg_std': frame[num_cols].std().mean(),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,method,remaining_missing,avg_mean,avg_std
0,mean,0,462.558776,137.009400
1,median,0,456.168950,137.433781
2,knn_k3,0,461.216101,148.259312
3,knn_k5,0,461.536389,147.948884


In [5]:
# Plot three representative variables and keep the stats table for the report.
selected_variables = ['CO(GT)', 'C6H6(GT)', 'T']
stats = compare_distributions(
    original=df,
    imputed_dict={'mean': baseline_mean, 'median': baseline_median, **imputed},
    columns=selected_variables,
    save_dir=FIGURES_DIR,
)
display(stats)

,variable,dataset,mean,median,std,min,max,iqr
0,CO(GT),original,2.152750,1.800000,1.453252,0.1,11.9,1.80
1,CO(GT),mean,2.152750,2.152750,1.308123,0.1,11.9,1.40
2,CO(GT),median,2.085820,1.800000,1.315415,0.1,11.9,1.40
3,CO(GT),knn_k3,2.088363,1.800000,1.410278,0.1,11.9,1.70
4,CO(GT),knn_k5,2.088283,1.800000,1.409529,0.1,11.9,1.66
5,C6H6(GT),original,10.083105,8.200000,7.449820,0.1,63.7,9.60
6,C6H6(GT),mean,10.083105,8.700000,7.258562,0.1,63.7,8.90
7,C6H6(GT),median,9.987668,8.200000,7.270307,0.1,63.7,8.90
8,C6H6(GT),knn_k3,10.149124,8.400000,7.413745,0.1,63.7,9.40
9,C6H6(GT),knn_k5,10.149996,8.400000,7.410725,0.1,63.7,9.40


## Guiding Questions

1. **Which imputation method preserves the original distribution more clearly?**

   KNN imputation usually preserves the original distribution more clearly than single-value imputation because it fills missing values using similar observations instead of replacing them with one fixed statistic.

2. **Did KNN imputation change the variability or shape of the selected variables?**

   Yes, but only moderately. KNN can smooth the tails and slightly reduce variability, although it normally keeps the overall shape and relationships between variables better than mean or median imputation.

3. **What value of k produced the most reasonable results? Justify your answer.**

   A value of k = 5 is a reasonable choice because it balances local detail and smoothing. Smaller values can make the result noisier, while larger values may over-smooth the data.